

---


# **Mushroom Identification - YOLO10 and DINOv2 Training**


---

This notebook trains on two new models on the mushroom dataset:


*   YOLOv10n-cls - Ultralytics NMS Free classification backbone.
*   DINOv2-small - Facebook self supervised ViT-s/14 backbone

Both follow the exact same training methodology as the existing models in repos (same argumentations, loss functions, optimizer, early stopping).









---


# **0. Verify GPU**

---



In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [14]:
!pip install ultralytics
!pip install split-folders
!pip install kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 53.4 MB/s eta 0:00:00




---


# **1.Clone the repo and check out banch**

---



In [2]:
!git clone https://github.com/QuasimodoCodes/Mushrooms.git
%cd Mushrooms
!git checkout

Cloning into 'Mushrooms'...
remote: Enumerating objects: 849, done.
remote: Counting objects: 100% (197/197), done.
remote: Compressing objects: 100% (153/153), done.
remote: Total 849 (delta 65), reused 137 (delta 34), pack-reused 652 (from 2)
Receiving objects: 100% (849/849), 214.64 MiB | 19.67 MiB/s, done.
Resolving deltas: 100% (334/334), done.
Updating files: 100% (190/190), done.
/content/Mushrooms
Your branch is up to date with 'origin/master'.


In [6]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("zlatan599/mushroom1")
print("Path to dataset files:", path)

100%|██████████| 11.3G/11.3G [11:14<00:00, 18.0MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/zlatan599/mushroom1/versions/2


In [3]:
import os
print("path")
!ls

path
config.py  docs       launch.py       requirements.txt	yolo26n-cls.pt
data	   dvc_plots  prometheus.yml  scripts
deploy	   dvc.yaml   README.md       services


In [7]:
!pip install -q split-folders

import splitfolders, shutil, os

source = "/root/.cache/kagglehub/datasets/zlatan599/mushroom1/versions/2/merged_dataset"
output = "data/dataset_split"

if os.path.exists(output):
    shutil.rmtree(output)

print("Splitting 80/10/10 — takes a few minutes...")
splitfolders.ratio(source, output=output, seed=42, ratio=(0.8, 0.1, 0.1))
print("Done!")

Splitting 80/10/10 — takes a few minutes...


Copying files: 104088 files [03:07, 554.26 files/s]

Done!


In [8]:
# Data splits
import os
for split in ["train", "val", "test"]:
    path = f"data/dataset_split/{split}"
    n_classes = len(os.listdir(path))
    n_images = sum(len(f) for _, _, f in os.walk(path))
    print(f"{split}: {n_classes} classes, {n_images:,} images")

train: 169 classes, 83,202 images
val: 169 classes, 10,337 images
test: 169 classes, 10,549 images




---


# **2. Train YOLOv10**


---



In [9]:
!git pull origin master

From https://github.com/QuasimodoCodes/Mushrooms
 * branch            master     -> FETCH_HEAD
Already up to date.


In [10]:
!ls scripts/training/

cnn		convnext  franken  yolo    yolov8n-cls.pt
compare_all.py	dinov2	  vit	   yolo10


In [11]:
!git branch
!git log --oneline -5

* master
ad22d17 (HEAD -> master, origin/master, origin/HEAD, origin/Chonthichar) Merge branch 'master' of https://github.com/QuasimodoCodes/Mushrooms
01ea01b feat: add YOLOv10 and DINOv2 training scripts
75f93ac Merge pull request #7 from QuasimodoCodes/feature/convnext-experiments
3421031 (origin/feature/convnext-experiments) Add ConvNeXt-Tiny training results, benchmark, and docs
acc412f feat: add YOLOv10 and DINOv2 training scripts


In [21]:
from ultralytics import YOLO
model = YOLO("yolov10n.pt")  # download yolov10n first

In [27]:
with open('scripts/training/yolo10/train_yolo10.py', 'r') as f:
    content = f.read()

content = content.replace('yolov8n-cls.pt', 'yolo11n-cls.pt')

with open('scripts/training/yolo10/train_yolo10.py', 'w') as f:
    f.write(content)

print("Fixed!")

Fixed!


In [ ]:
!python scripts/training/yolo10/train_yolo10.py

  Starting YOLOv10 Classification Training
>> SUCCESS: GPU detected — Tesla T4

Loading YOLOv10n-cls pretrained weights...

Training on dataset: /content/Mushrooms/data/dataset_split
Watch 'loss' decrease — early stopping triggers after 10 flat epochs.

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/Mushrooms/data/dataset_split, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, 